In [1]:
import pyrosetta
import numpy as np

from benchmark import bpti_ryfyn_benchmark
from energy_calculation import evaluate_quantum_energies, evaluate_pyrosetta_energies, compare_energies
from misc import init_generator_params
from rotamer_extraction import extract_top_n_rotamers
from h_ising_creation import extract_hamiltonian_tensors, build_ising_hamiltonian, reduce_hamiltonian
from initialisation import initialize_rosetta
from custom_qaoa import qaoa_func_generator, run_qaoa
from h_mixer import custom_xy_mixer_layer

from constants import *
from validation import validate_conformations

initialize_rosetta(pyrosetta, extra_flags="-mute all")

Initializing PyRosetta with cleaning flags: -ignore_unrecognized_res and extra flags: -mute all
┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python312.ubuntu 2026.03+releasequarterly.5e498f1409c68ade56c8ce5842bf79e1b02e8db4 2026-01-13T13:24:11] retrieved from: http://www.pyrosetta.org


In [2]:
# Pyrosetta Relevant Code
benchmark_pose = bpti_ryfyn_benchmark()
rotamer_lib, ig, rot_sets, scorefxn = extract_top_n_rotamers(benchmark_pose)

# Generating QUBO (Quadratic Unconstrained Binary Optimisation) Model, and then reduce it
h_linear, J_quadratic = extract_hamiltonian_tensors(rotamer_lib, ig, rot_sets)
h_flex_linear, J_flex_quadratic, global_offset = reduce_hamiltonian(h_linear, J_quadratic, rotamer_lib)
generator_params = init_generator_params(h_flex_linear)
for idx in h_linear:
    print(h_linear[idx])
    print(h_flex_linear.get(idx, "None"))
    print("\n---------------------------------------\n")

# Generate the actual observable and running functions we will use in the QAOA Algorithm
H_ising = build_ising_hamiltonian(h_flex_linear, J_flex_quadratic, global_offset, penalty=0.0)
cost_function, sample_function = qaoa_func_generator(H_ising, custom_xy_mixer_layer, generator_params)

# Run the Quantum Approximate Optimisation Algorithm and sample the final parameters
final_params = run_qaoa(cost_function)
probabilities = sample_function(final_params)

# Extract the top 100 most probably conformations and check that exactly 1 rotamer for each residue is selected
top_indices = list(np.argsort(probabilities)[-TOP_CONFORMATION_COUNTS:][::-1])
valid_conformations = validate_conformations(top_indices, probabilities, generator_params)

Fragment Sequence: RYFYN
Total Residues: 5
Creating score function
Pose scored successfully!
Creating Repacking Task - Core Rotamer Optimisation Protocol
Computing One-Body and Two-Body Energies
Iterating through molten residues - determining the top rotamer positions for each amino acid
1 1
2 2
3 3
4 4
5 5
initializing generator_params
{0: 1.0419909954071045, 1: 1.2052754163742065, 2: 1.2125394344329834, 3: 1.6054599285125732}
{0: 1.0419909954071045, 1: 1.2052754163742065, 2: 1.2125394344329834, 3: 1.6054599285125732}

---------------------------------------

{0: 1.4519609212875366, 1: 1.4519609212875366, 2: 2.4010090827941895, 3: 2.4010090827941895}
{0: 1.4519609212875366, 1: 1.4519609212875366, 2: 2.4010090827941895, 3: 2.4010090827941895}

---------------------------------------

{0: 1.4840223789215088, 1: 1.6721895933151245, 2: 2.909672975540161}
{0: 1.4840223789215088, 1: 1.6721895933151245, 2: 2.909672975540161}

---------------------------------------

{0: 1.8487499952316284, 1

In [3]:
def evaluate_quantum_energies(valid_conformations, h_flex, J_flex, global_offset, params):
    wire_offsets = params["wire_offsets"]

    for conformation in valid_conformations:
        bitstring = conformation["bitstring"]
        current_energy = global_offset

        # One body energies
        for seq, energies in h_flex.items():
            base_wire = wire_offsets[seq]
            for rot, e_val in enumerate(energies):
                if bitstring[base_wire + rot] == 1:
                    current_energy += e_val

        # Two body energies
        for (seq_i, seq_j), interactions in J_flex.items():
            for (rot_i, rot_j), e_val in interactions.items():
                k = wire_offsets[seq_i] + rot_i
                l = wire_offsets[seq_j] + rot_j

                if bitstring[k] == 1 and bitstring[l] == 1:
                    current_energy += e_val

        conformation['quantum_energy'] = current_energy
    # raise NotImplementedError("Not yet implemented")

evaluate_quantum_energies(valid_conformations, h_flex_linear, J_flex_quadratic, global_offset, params=generator_params)
valid_conformations.sort(key=lambda conf: conf['quantum_energy'])
print(valid_conformations)

[{'bitstring': [0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0], 'probability': np.float64(1.0769018313700486e-05), 'quantum_energy': -7.076041080057621, 'biological_energy': None, 'pose': None}, {'bitstring': [0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0], 'probability': np.float64(1.7029015136209383e-05), 'quantum_energy': -6.53022513538599, 'biological_energy': None, 'pose': None}, {'bitstring': [1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0], 'probability': np.float64(0.9999181148124267), 'quantum_energy': -6.330225087702274, 'biological_energy': None, 'pose': None}, {'bitstring': [0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0], 'probability': np.float64(6.108059980847955e-12), 'quantum_energy': -6.076041080057621, 'biological_energy': None, 'pose': None}, {'bitstring': [0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0], 'probability': np.float64(6.054577842626313e-12), 'quantum_energy': -6.075305320322514, 'biological_energy': None, '

In [4]:
def evaluate_pyrosetta_energies(pose, valid_conformations, scorefxn, rotamer_library, params):
    for conformation in valid_conformations:

        bitstring = conformation["bitstring"]
        bio_energy, pose = evaluate_singular_pyrosetta_energy(pose, bitstring, scorefxn, rotamer_library, params)
        conformation['biological_energy'] = bio_energy
        conformation['pose'] = pose


def evaluate_singular_pyrosetta_energy(pose, bitstrings, scorefxn, rotamer_library, params):
    test_pose = pose.clone()

    seq_positions = params["seq_positions"]
    wire_offsets = params["wire_offsets"]
    rotamer_counts = params["rotamer_counts"]

    for seq in seq_positions:
        base_wire = wire_offsets[seq]
        num_rots = rotamer_counts[seq]

        residue_bits = bitstrings[base_wire : base_wire + num_rots]
        local_rotamer_idx = residue_bits.index(1)

        res_entry = rotamer_library[seq]
        rotamer_entry = res_entry[local_rotamer_idx]
        selected_residue = rotamer_entry[2]

        test_pose.replace_residue(seq, selected_residue, False)

    bio_energy = scorefxn(test_pose)
    return bio_energy, test_pose

evaluate_pyrosetta_energies(benchmark_pose, valid_conformations, scorefxn, rotamer_lib, generator_params)

In [5]:
deltas = [conf['quantum_energy'] - conf['biological_energy'] for conf in valid_conformations]
deltas

[-6.535111994133817,
 -5.542375590513398,
 -7.371827561936344,
 -5.535111994133817,
 -5.535111866824853,
 -5.661650258335067,
 -4.542375590513398,
 -4.542375463204435,
 -4.668913854714649,
 -6.371827561936344,
 -6.371827434627381,
 -6.498365826137595,
 -4.9184870979442294,
 -5.484159982330587,
 -5.960762810879517,
 -4.9352967163914565,
 -3.9257506943238107,
 -4.491423663453279,
 -5.755202665746757,
 -5.371827434627381,
 -6.32087565911797,
 -5.498365826137595,
 -5.498365698828632,
 -4.968026393807376,
 -6.797478359977443,
 -4.723297953173237,
 -4.484159982330587,
 -3.9352967163914565,
 -3.935296589082493,
 -4.061834980592708,
 -4.47030477162075,
 -3.7305615495528173,
 -3.491423663453279,
 -4.755202665746757,
 -4.755202538437794,
 -5.5600135209757635,
 -5.32087565911797,
 -5.320875531809007,
 -5.797478359977443,
 -5.79747823266848,
 -5.447413923319219,
 -5.9240166476273775,
 -3.477568368000332,
 -3.723297953173237,
 -5.307020339423278,
 -3.31867182020187,
 -3.8843448639482263,
 -2.730561

In [6]:
valid_conformations

[{'bitstring': [0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
  'probability': np.float64(1.0769018313700486e-05),
  'quantum_energy': -7.076041080057621,
  'biological_energy': -0.5409290859238043,
  'pose': <pyrosetta.rosetta.core.pose.Pose at 0x71f2fcdeb430>},
 {'bitstring': [0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
  'probability': np.float64(1.7029015136209383e-05),
  'quantum_energy': -6.53022513538599,
  'biological_energy': -0.9878495448725921,
  'pose': <pyrosetta.rosetta.core.pose.Pose at 0x71f2fcde87b0>},
 {'bitstring': [1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
  'probability': np.float64(0.9999181148124267),
  'quantum_energy': -6.330225087702274,
  'biological_energy': 1.04160247423407,
  'pose': <pyrosetta.rosetta.core.pose.Pose at 0x71f2fcdea4b0>},
 {'bitstring': [0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
  'probability': np.float64(6.108059980847955e-12),
  'quantum_energy': -6.076041080057621,
  'biological_

In [7]:
bio_energy = [conf['biological_energy'] for conf in valid_conformations]
quant_energy = [conf['quantum_energy'] for conf in valid_conformations]

bio_ordering =  [int(idx) for idx in list(np.argsort(bio_energy))]
quant_ordering = [int(idx) for idx in list(np.argsort(quant_energy))]

for bio_idx, quant_idx in zip(bio_ordering, quant_ordering):
    print(f"Bio Ranking: Quantum Ranking | {bio_idx}: {quant_idx}")

Bio Ranking: Quantum Ranking | 1: 0
Bio Ranking: Quantum Ranking | 6: 1
Bio Ranking: Quantum Ranking | 7: 2
Bio Ranking: Quantum Ranking | 8: 3
Bio Ranking: Quantum Ranking | 16: 4
Bio Ranking: Quantum Ranking | 3: 5
Bio Ranking: Quantum Ranking | 0: 6
Bio Ranking: Quantum Ranking | 4: 7
Bio Ranking: Quantum Ranking | 5: 8
Bio Ranking: Quantum Ranking | 12: 9
Bio Ranking: Quantum Ranking | 17: 10
Bio Ranking: Quantum Ranking | 32: 11
Bio Ranking: Quantum Ranking | 27: 12
Bio Ranking: Quantum Ranking | 15: 13
Bio Ranking: Quantum Ranking | 28: 14
Bio Ranking: Quantum Ranking | 31: 15
Bio Ranking: Quantum Ranking | 47: 16
Bio Ranking: Quantum Ranking | 29: 17
Bio Ranking: Quantum Ranking | 42: 18
Bio Ranking: Quantum Ranking | 26: 19
Bio Ranking: Quantum Ranking | 13: 20
Bio Ranking: Quantum Ranking | 45: 21
Bio Ranking: Quantum Ranking | 25: 22
Bio Ranking: Quantum Ranking | 43: 23
Bio Ranking: Quantum Ranking | 23: 24
Bio Ranking: Quantum Ranking | 30: 25
Bio Ranking: Quantum Ranking |

In [8]:
rank_mismatch = [abs(a-b) for a, b in zip(bio_ordering, quant_ordering)]
print("Mean rank mismatch", np.mean(rank_mismatch))
print("Median rank mismatch", np.median(rank_mismatch))

print("Mean rank mismatch of first 10", np.mean(rank_mismatch[:10]))
print("Median rank mismatch of first 10", np.median(rank_mismatch[:10]))

print("Mean rank mismatch of last 10", np.mean(rank_mismatch[-10:]))
print("Median rank mismatch of last 10", np.median(rank_mismatch[-10:]))

Mean rank mismatch 9.16
Median rank mismatch 6.5
Mean rank mismatch of first 10 4.5
Median rank mismatch of first 10 4.0
Mean rank mismatch of last 10 0.2
Median rank mismatch of last 10 0.0
